#### Предсказываем цену ноутбука

В папке лежит датасет laptop_prices, в котором содержатся сведения о ноутбуках, их характеристики и цены. Обучите модель линейной регрессии, которая будет предсказывать цену ноутбука по его характеристикам. 

Придется хорошенько поработать с характеристиками: это *творческая* часть задания. Во-первых, надо привести их в машиночитаемый вид, а во-вторых, можно посмотреть, как они коррелируют друг с другом и не нужно ли кого-то из них дропнуть или наоборот. Не советую бездумно использовать OHE: некоторые признаки явно можно закодировать куда более оптимальным способом. 

*Примечание*: без работы над фичами за все дз - **0 баллов**. 

In [193]:
import pandas as pd

data = pd.read_csv('/Users/katyamazurina/Desktop/2 sem/ML/laptop_prices.csv')

In [194]:
data. head()

,Brand,Processor,RAM (GB),Storage,GPU,Screen Size (inch),Resolution,Battery Life (hours),Weight (kg),Operating System,Price ($)
0,Apple,AMD Ryzen 3,64,512GB SSD,Nvidia GTX 1650,17.3,2560x1440,8.9,1.42,FreeDOS,3997.07
1,Razer,AMD Ryzen 7,4,1TB SSD,Nvidia RTX 3080,14.0,1366x768,9.4,2.57,Linux,1355.78
2,Asus,Intel i5,32,2TB SSD,Nvidia RTX 3060,13.3,3840x2160,8.5,1.74,FreeDOS,2673.07
3,Lenovo,Intel i5,4,256GB SSD,Nvidia RTX 3080,13.3,1366x768,10.5,3.10,Windows,751.17
4,Razer,Intel i3,4,256GB SSD,AMD Radeon RX 6600,16.0,3840x2160,5.7,3.38,Linux,2059.83


In [195]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11768 entries, 0 to 11767
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 11768 non-null  object 
 1   Processor             11768 non-null  object 
 2   RAM (GB)              11768 non-null  int64  
 3   Storage               11768 non-null  object 
 4   GPU                   11768 non-null  object 
 5   Screen Size (inch)    11768 non-null  float64
 6   Resolution            11768 non-null  object 
 7   Battery Life (hours)  11768 non-null  float64
 8   Weight (kg)           11768 non-null  float64
 9   Operating System      11768 non-null  object 
 10  Price ($)             11768 non-null  float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1011.4+ KB


In [201]:
# количество уникальных значений 
unique_data = data.nunique()
unique_data

Brand                      10
Processor                   8
RAM (GB)                    5
Storage                     5
GPU                         7
Screen Size (inch)          5
Resolution                  4
Battery Life (hours)       81
Weight (kg)               231
Operating System            4
Price ($)               11558
dtype: int64

In [202]:
import re

def storage_type(Storage): # из 'storage' выделим 'HDD/SSD' в отдельный признак
    return 'SSD' if 'SSD' in Storage else 'HDD' if 'HDD' in Storage else 'None'
data['Type_of_Storage'] = data.Storage.apply(storage_type)
data.Storage = data.Storage.apply(lambda x: re.sub(r'[A-Za-z]', '', str(x))) 
data.Storage = data.Storage.apply(lambda x: int(x) * 1024 if int(x) == 1 or int(x) == 2 else int(x) * 1) # переводим TB в GB

data.Processor = data.Processor.apply(lambda x: re.sub(r'[A-Za-z]', '', str(x))) # оставляю только числовые значения

data = pd.get_dummies(data, columns=['Type_of_Storage'], dtype=int)
data = pd.get_dummies(data, columns=['Brand'], dtype=int)

data.drop(columns=['GPU'], inplace=True)
data.drop(columns=['Battery Life (hours)'], inplace=True)
data.drop(columns=['Operating System'], inplace=True)
data.drop(columns=['Weight (kg)'], inplace=True)

def resolution_split(resolution): # разделим разрешения на ширину и высоту 
    width, height = resolution.split('x')
    return int(width), int(height)

# вместо 'resolution' два столбца со значениями ширины и высоты 
data[['Width', 'Height']] = data.Resolution.apply(lambda x: pd.Series(resolution_split(x))) 
data = data.drop(columns=['Resolution'])

In [203]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=['Price ($)'])  
y = data['Price ($)']

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

In [204]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()
model.fit(Xtrain, ytrain) 

pred_train = model.predict(Xtrain)
pred_test = model.predict(Xtest)

In [205]:
mean_squared_error(pred_test, ytest) ** 0.5

549.5091207513998

In [206]:
mean_squared_error(pred_train, ytrain) ** 0.5

524.675751987633